<a href="https://colab.research.google.com/github/aniolarolas/quant-research-pricing-credit-risk/blob/task-02-contract-pricing/02_storage_contract_pricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: Price a commodity storage contract

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Problem Understanding

The objective of this task is to build a prototype pricing function for a natural gas storage contract.

The client wants to buy gas when prices are relatively low, inject it into storage, and withdraw it later when prices are expected to be higher. The value of the contract depends on the difference between the selling price and the buying price, minus all costs required to execute the strategy.

The model should account for:

- injection dates,
- withdrawal dates,
- gas prices on those dates,
- injection and withdrawal rates,
- maximum storage capacity,
- storage costs,
- injection and withdrawal costs.

Since interest rates are assumed to be zero, future cash flows do not need to be discounted.

## 2. Contract Valuation Logic

A natural gas storage contract can be valued by looking at all cash flows generated by the strategy.

The client injects gas into storage on selected dates. On those dates, the client effectively buys gas, so this creates a cash outflow. Later, the client withdraws gas from storage and sells it on selected withdrawal dates, creating a cash inflow.

The value of the contract is therefore:

$$
\text{Contract Value}
=
\text{Sales Revenue}
-
\text{Purchase Cost}
-
\text{Injection Costs}
-
\text{Withdrawal Costs}
-
\text{Storage Costs}
$$

where:

- **Sales Revenue** is the money received from selling gas on withdrawal dates.
- **Purchase Cost** is the cost of buying gas on injection dates.
- **Injection Costs** are costs paid when gas is injected into storage.
- **Withdrawal Costs** are costs paid when gas is withdrawn from storage.
- **Storage Costs** are costs paid for keeping gas in storage over time.

Since interest rates are assumed to be zero, future cash flows are not discounted.

## 3. Mathematical Formulation

Let the contract have a set of injection dates and withdrawal dates.

For each injection date $t_i$, the client injects a volume $q_i^{in}$ of gas. The cost of buying and injecting gas is:

$$
\text{Injection Cash Flow}_i =
P(t_i) \cdot q_i^{in}
+
c^{in} \cdot q_i^{in}
$$

where:

- $P(t_i)$ is the estimated gas price on the injection date,
- $q_i^{in}$ is the injected volume,
- $c^{in}$ is the injection cost per unit.

For each withdrawal date $t_j$, the client withdraws and sells a volume $q_j^{out}$ of gas. The revenue net of withdrawal costs is:

$$
\text{Withdrawal Cash Flow}_j =
P(t_j) \cdot q_j^{out}
-
c^{out} \cdot q_j^{out}
$$

where:

- $P(t_j)$ is the estimated gas price on the withdrawal date,
- $q_j^{out}$ is the withdrawn volume,
- $c^{out}$ is the withdrawal cost per unit.

The total value of the contract is:

$$
V =
\sum_j
\left(
P(t_j) \cdot q_j^{out}
-
c^{out} \cdot q_j^{out}
\right)
-
\sum_i
\left(
P(t_i) \cdot q_i^{in}
+
c^{in} \cdot q_i^{in}
\right)
-
C^{storage}
$$

The storage cost is treated as an additional cost of holding gas in the storage facility during the life of the contract.

## 4. Cash Flow Helper Functions

The contract value is built from three types of cash flows:

- injection cash flows,
- withdrawal cash flows,
- storage costs.

To keep the final pricing function readable, each component is calculated using a small helper function.

### Calculate months between injection and withdraw

In [45]:
def months_between(start_date, end_date):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    return (end_date - start_date).days / 30.4375

### Calculate injection/withdraw costs

In [27]:
def calculate_injection_cashflow(price, volume, injection_cost_rate):
  purchase_cost = price * volume
  injection_cost = injection_cost_rate * volume
  return purchase_cost + injection_cost

def calculate_withdrawal_cashflow(price, volume, withdrawal_cost_rate):
    sales_revenue = price * volume
    withdrawal_cost = withdrawal_cost_rate * volume
    return sales_revenue - withdrawal_cost

### Calculate storage costs

In [28]:
def calculate_storage_cost(inventory, months_held, storage_cost_per_unit_per_month):
    return inventory * months_held * storage_cost_per_unit_per_month

### Create events table

Instead of working directly with separate lists of injection and withdrawal dates, the contract is converted into a single chronological events table.

Each row represents one relevant contract date and contains:

- the volume injected on that date,
- the volume withdrawn on that date.

This structure makes the pricing logic easier to implement, because the contract can be processed date by date while tracking the gas inventory over time.

In [29]:
def build_events_table(
    injection_dates,
    withdrawal_dates,
    injection_volumes,
    withdrawal_volumes
):
    injections = pd.DataFrame({
        "date": pd.to_datetime(injection_dates),
        "injected_volume": injection_volumes,
        "withdrawn_volume": 0.0
    })

    withdrawals = pd.DataFrame({
        "date": pd.to_datetime(withdrawal_dates),
        "injected_volume": 0.0,
        "withdrawn_volume": withdrawal_volumes
    })

    events = pd.concat([injections, withdrawals])

    events = (
        events
        .groupby("date", as_index=False)[["injected_volume", "withdrawn_volume"]]
        .sum()
        .sort_values("date")
        .reset_index(drop=True)
    )

    return events

### Events Table Test

In [30]:
injection_dates = ["2024-06-01", "2024-07-01"]
withdrawal_dates = ["2024-12-01", "2025-01-01"]

injection_volumes = [1_000_000, 1_000_000]
withdrawal_volumes = [1_000_000, 1_000_000]

events_test = build_events_table(
    injection_dates,
    withdrawal_dates,
    injection_volumes,
    withdrawal_volumes
)

events_test

,date,injected_volume,withdrawn_volume
0,2024-06-01,1000000.0,0.0
1,2024-07-01,1000000.0,0.0
2,2024-12-01,0.0,1000000.0
3,2025-01-01,0.0,1000000.0


### Gas Price Estimation Model

This storage contract pricing model depends on the gas price estimation function developed in Task 1.

In Task 1, we built a simple time series model that estimates the natural gas price for any given date using:

- a linear time trend,
- annual seasonality represented with sine and cosine terms.

In this task, the function `estimate_gas_price()` is used as an input to the storage valuation model. For each injection or withdrawal date, the contract pricing function calls `estimate_gas_price()` to obtain the estimated market price of natural gas on that date.

To keep this notebook self-contained and executable independently, the relevant model setup from Task 1 is reproduced below.

In [31]:
# Load and train the gas price model from Task 1
df = pd.read_csv("/content/drive/MyDrive/quant_research_jpm_jobsim/data/raw/task01/Nat_Gas.csv")

df["Dates"] = pd.to_datetime(df["Dates"], format="%m/%d/%y")
df = df.sort_values("Dates")

df["time_index"] = np.arange(len(df))
df["sin_12"] = np.sin(2 * np.pi * df["time_index"] / 12)
df["cos_12"] = np.cos(2 * np.pi * df["time_index"] / 12)

X = df[["time_index", "sin_12", "cos_12"]]
y = df["Prices"]

model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [32]:
def estimate_gas_price(input_date):
    input_date = pd.to_datetime(input_date)
    first_date = df["Dates"].min()

    t = (input_date - first_date).days / 30.4375

    if 0 <= t <= 60:
        sin_12 = np.sin(2 * np.pi * t / 12)
        cos_12 = np.cos(2 * np.pi * t / 12)

        input_features = pd.DataFrame({
            "time_index": [t],
            "sin_12": [sin_12],
            "cos_12": [cos_12]
        })

        estimated_price = model.predict(input_features)[0]
        return estimated_price
    else:
        return "Date out of range"

## 5. Storage Contract Pricing Function

The final function processes the contract chronologically using the events table.

For each contract date, the function:

1. estimates the gas price on that date,
2. applies injection cash flows,
3. applies withdrawal cash flows,
4. updates the inventory,
5. checks operational constraints,
6. charges storage costs until the next contract date.

The function returns both the final contract value and a breakdown of the main cash flow components.

The main inputs are:

- `injection_dates`: dates when gas is bought and injected into storage.
- `withdrawal_dates`: dates when gas is withdrawn and sold.
- `injection_volumes`: volumes injected on each injection date.
- `withdrawal_volumes`: volumes withdrawn on each withdrawal date.
- `injection_cost_rate`: cost per unit injected.
- `withdrawal_cost_rate`: cost per unit withdrawn.
- `storage_cost_per_unit_per_month`: monthly storage cost per unit of gas held in storage.
- `max_storage_volume`: maximum amount of gas that can be stored.
- `injection_rate`: maximum amount that can be injected on a single injection date.
- `withdrawal_rate`: maximum amount that can be withdrawn on a single withdrawal date.

The function should compute all purchase costs, sales revenues, operational costs, and storage costs. It should also check that the contract respects the basic operational constraints of the storage facility.

In [34]:
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    injection_volumes,
    withdrawal_volumes,
    injection_cost_rate,
    withdrawal_cost_rate,
    storage_cost_per_unit_per_month,
    max_storage_volume,
    injection_rate,
    withdrawal_rate,
    price_function=None
):
    if price_function is None:
        price_function = estimate_gas_price

    events = build_events_table(
        injection_dates,
        withdrawal_dates,
        injection_volumes,
        withdrawal_volumes
    )

    inventory = 0.0
    contract_value = 0.0

    total_purchase_cost = 0.0
    total_sales_revenue = 0.0
    total_injection_cost = 0.0
    total_withdrawal_cost = 0.0
    total_storage_cost = 0.0

    details = []

    for i in range(len(events)):
        current_date = events.loc[i, "date"]
        injected_volume = events.loc[i, "injected_volume"]
        withdrawn_volume = events.loc[i, "withdrawn_volume"]

        price = price_function(current_date)

        if isinstance(price, str):
            raise ValueError(f"Price could not be estimated for date {current_date}: {price}")

        # Injection
        if injected_volume > 0:
            if injected_volume > injection_rate:
                raise ValueError(f"Injection rate exceeded on {current_date}")

            purchase_cost = price * injected_volume
            injection_cost = injection_cost_rate * injected_volume

            contract_value -= purchase_cost
            contract_value -= injection_cost

            total_purchase_cost += purchase_cost
            total_injection_cost += injection_cost

            inventory += injected_volume

            if inventory > max_storage_volume:
                raise ValueError(f"Maximum storage capacity exceeded on {current_date}")

        # Withdrawal
        if withdrawn_volume > 0:
            if withdrawn_volume > withdrawal_rate:
                raise ValueError(f"Withdrawal rate exceeded on {current_date}")

            if withdrawn_volume > inventory:
                raise ValueError(f"Cannot withdraw more gas than available on {current_date}")

            sales_revenue = price * withdrawn_volume
            withdrawal_cost = withdrawal_cost_rate * withdrawn_volume

            contract_value += sales_revenue
            contract_value -= withdrawal_cost

            total_sales_revenue += sales_revenue
            total_withdrawal_cost += withdrawal_cost

            inventory -= withdrawn_volume

        # Storage cost until next event
        storage_cost = 0.0
        months_held = 0.0

        if i < len(events) - 1:
            next_date = events.loc[i + 1, "date"]
            months_held = months_between(current_date, next_date)

            storage_cost = calculate_storage_cost(
                inventory,
                months_held,
                storage_cost_per_unit_per_month
            )

            contract_value -= storage_cost
            total_storage_cost += storage_cost

        details.append({
            "date": current_date,
            "price": price,
            "injected_volume": injected_volume,
            "withdrawn_volume": withdrawn_volume,
            "inventory_after_event": inventory,
            "months_until_next_event": months_held,
            "storage_cost_until_next_event": storage_cost,
            "contract_value_after_event": contract_value
        })

    details_df = pd.DataFrame(details)

    result = {
        "contract_value": contract_value,
        "total_purchase_cost": total_purchase_cost,
        "total_sales_revenue": total_sales_revenue,
        "total_injection_cost": total_injection_cost,
        "total_withdrawal_cost": total_withdrawal_cost,
        "total_storage_cost": total_storage_cost,
        "final_inventory": inventory,
        "details": details_df
    }

    return result

### Base Case Valuation

In [38]:
result = price_storage_contract(
    injection_dates=["2024-06-01", "2024-07-01"],
    withdrawal_dates=["2024-12-01", "2025-01-01"],
    injection_volumes=[1_000_000, 1_000_000],
    withdrawal_volumes=[1_000_000, 1_000_000],
    injection_cost_rate=0.01,
    withdrawal_cost_rate=0.01,
    storage_cost_per_unit_per_month=0.10,
    max_storage_volume=2_000_000,
    injection_rate=1_000_000,
    withdrawal_rate=1_000_000
)

result["contract_value"]
result["details"]


,date,price,injected_volume,withdrawn_volume,inventory_after_event,months_until_next_event,storage_cost_until_next_event,contract_value_after_event
0,2024-06-01,11.782824,1000000.0,0.0,1000000.0,0.985626,9.856263e+04,-1.189139e+07
1,2024-07-01,11.564958,1000000.0,0.0,2000000.0,5.026694,1.005339e+06,-2.447168e+07
2,2024-12-01,12.689519,0.0,1000000.0,1000000.0,1.018480,1.018480e+05,-1.189401e+07
3,2025-01-01,13.003690,0.0,1000000.0,0.0,0.000000,0.000000e+00,1.099677e+06


In [40]:
summary = pd.DataFrame({
    "Component": [
        "Sales Revenue",
        "Purchase Cost",
        "Injection Cost",
        "Withdrawal Cost",
        "Storage Cost",
        "Contract Value",
        "Final Inventory"
    ],
    "Value": [
        result["total_sales_revenue"],
        -result["total_purchase_cost"],
        -result["total_injection_cost"],
        -result["total_withdrawal_cost"],
        -result["total_storage_cost"],
        result["contract_value"],
        result["final_inventory"]
    ]
})

display_summary = summary.copy()
display_summary["Value"] = display_summary["Value"].map(lambda x: f"{x:,.2f}")
display_summary

,Component,Value
0,Sales Revenue,"25,693,209.05"
1,Purchase Cost,"-23,347,782.08"
2,Injection Cost,"-20,000.00"
3,Withdrawal Cost,"-20,000.00"
4,Storage Cost,"-1,205,749.49"
5,Contract Value,"1,099,677.49"
6,Final Inventory,0.00


## 6. Constraint Tests

The pricing function should not only calculate the value of valid contracts, but also reject contracts that violate basic storage constraints.

The following tests check that the function correctly raises errors when:

- the injection rate is exceeded,
- the withdrawal rate is exceeded,
- the maximum storage capacity is exceeded,
- the client tries to withdraw more gas than is available in storage.

Injection rate exceeded

In [41]:
try:
    price_storage_contract(
        injection_dates=["2024-06-01"],
        withdrawal_dates=["2024-12-01"],
        injection_volumes=[2_000_000],
        withdrawal_volumes=[2_000_000],
        injection_cost_rate=0.01,
        withdrawal_cost_rate=0.01,
        storage_cost_per_unit_per_month=0.10,
        max_storage_volume=2_000_000,
        injection_rate=1_000_000,
        withdrawal_rate=2_000_000
    )
except ValueError as e:
    print("Error correctly detected:", e)

Error correctly detected: Injection rate exceeded on 2024-06-01 00:00:00


Withdrawal rate exceeded

In [42]:
try:
    price_storage_contract(
        injection_dates=["2024-06-01"],
        withdrawal_dates=["2024-12-01"],
        injection_volumes=[2_000_000],
        withdrawal_volumes=[2_000_000],
        injection_cost_rate=0.01,
        withdrawal_cost_rate=0.01,
        storage_cost_per_unit_per_month=0.10,
        max_storage_volume=2_000_000,
        injection_rate=2_000_000,
        withdrawal_rate=1_000_000
    )
except ValueError as e:
    print("Error correctly detected:", e)

Error correctly detected: Withdrawal rate exceeded on 2024-12-01 00:00:00


Maximum storage capacity exceeded

In [43]:
try:
    price_storage_contract(
        injection_dates=["2024-06-01"],
        withdrawal_dates=["2024-12-01"],
        injection_volumes=[2_000_000],
        withdrawal_volumes=[2_000_000],
        injection_cost_rate=0.01,
        withdrawal_cost_rate=0.01,
        storage_cost_per_unit_per_month=0.10,
        max_storage_volume=1_000_000,
        injection_rate=2_000_000,
        withdrawal_rate=2_000_000
    )
except ValueError as e:
    print("Error correctly detected:", e)

Error correctly detected: Maximum storage capacity exceeded on 2024-06-01 00:00:00


Withdrawing more gas than available

In [44]:
try:
    price_storage_contract(
        injection_dates=["2024-06-01"],
        withdrawal_dates=["2024-12-01"],
        injection_volumes=[1_000_000],
        withdrawal_volumes=[2_000_000],
        injection_cost_rate=0.01,
        withdrawal_cost_rate=0.01,
        storage_cost_per_unit_per_month=0.10,
        max_storage_volume=2_000_000,
        injection_rate=2_000_000,
        withdrawal_rate=2_000_000
    )
except ValueError as e:
    print("Error correctly detected:", e)

Error correctly detected: Cannot withdraw more gas than available on 2024-12-01 00:00:00


The constraint tests confirm that the pricing function rejects contracts that are not operationally feasible. This is important because a storage contract valuation should not only compute cash flows, but also ensure that the proposed injection and withdrawal schedule respects the physical limits of the storage facility.

## 7. Limitations and Next Steps

This prototype provides a transparent valuation framework for a natural gas storage contract. It combines estimated gas prices with the main contractual cash flows: purchase costs, sales revenues, injection and withdrawal costs, and storage costs.

The model also checks the main operational constraints of the storage facility, including injection rates, withdrawal rates, maximum storage capacity, and inventory availability.

However, the model is still a simplified prototype. It assumes zero interest rates, no transport delays, no holidays, and no uncertainty around future gas prices. The gas prices used in the valuation come from the estimation model developed in Task 1, so the contract value depends directly on the quality and limitations of that price model.

Future improvements could include discounting future cash flows, incorporating bid-ask spreads, modelling uncertainty in future gas prices, adding more detailed storage fees, and stress-testing the contract value under different market scenarios.

Overall, the function provides a clear and flexible starting point for pricing storage contracts before further validation, testing, and integration into a production environment.